# Global low-rank head: CODI versus explicit GPT-2 CoT, with full timing diagnostics

Run all cells with a Kaggle GPU and Internet. **No data attachments or old results are
needed.** The historical benchmark is unchanged; this notebook embeds its new runtime.

Both modes use the released CODI GPT-2 checkpoint. CODI performs six latent passes;
explicit CoT generates directly after the question using the shared teacher/student
weights. This is not a separately trained CoT-SFT checkpoint. A separate head is fitted
on each mode's own trajectories using activation whitening, nested ranks 32/64/96,
KL + top-token + margin losses, and compressed-policy recovery.

Deployment compares dense, eager rank 96, compiled rank 96, historical Triton arithmetic,
and FP32-accumulating Triton. Numerical parity is measured, never assumed.

1. **Clean free generation:** actual latency with each head's own outputs.
2. **Clean fixed replay:** identical dense-reference tokens and step counts for every
   head, including transformer/cache work, to isolate implementation efficiency.
3. **Detailed diagnostics:** every module, stage, transfer, synchronization, and
   underlying profiler operation on a smaller fixed sample. Instrumentation adds
   overhead, so these measurements stay separate from clean speedups.

The PDF's numbers are historical references, not expected outputs. Explicit traces can
be longer, but improved overall speed is a hypothesis.

In [ ]:
SMOKE=False  # optional quick validation; False runs real experiments
SEED=89
MODES=['codi','explicit_cot']
FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS=1024,256,256
MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES=4096,1024,2048
CLEAN_EPOCHS,RECOVERY_EPOCHS=4,2
DISTILL_BATCH_SIZE=8
COLLECT_BATCH_SIZE=8
QUALITY_BATCH_SIZE=16
MAX_NEW_TOKENS={'codi':64,'explicit_cot':256}
TIMING_QUESTIONS=64
TIMING_BATCH_SIZES=(1,8,32)
TIMING_REPEATS=5
PROFILE_QUESTIONS=4  # module/stage events at batch 1
PROFILE_REPEATS=2
OPERATOR_TRACE_QUESTIONS=1  # complete operator trace per mode/head
TRY_COMPILE=True
TRY_TRITON=True
QUALITY_QUESTIONS=1319
RANKS=(32,64,96)
RESUME_DIR=''  # optional matching prior output run folder
OUTPUT_ROOT='/kaggle/working/codi_explicit_global_head'
if SMOKE:
    FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS=16,8,8
    MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES=128,64,64
    CLEAN_EPOCHS=RECOVERY_EPOCHS=1
    TIMING_QUESTIONS,TIMING_REPEATS,QUALITY_QUESTIONS=4,2,8
    TIMING_BATCH_SIZES=(1,4)
    PROFILE_QUESTIONS,PROFILE_REPEATS=1,1
    MAX_NEW_TOKENS={'codi':16,'explicit_cot':32}
assert 1 in TIMING_BATCH_SIZES
assert 0 < PROFILE_QUESTIONS <= TIMING_QUESTIONS
assert 0 <= OPERATOR_TRACE_QUESTIONS <= TIMING_QUESTIONS
assert PROFILE_REPEATS > 0 and TIMING_REPEATS > 1
import gc,gzip,hashlib,importlib.util,json,os,pathlib,random,shutil,subprocess,sys,time
from contextlib import contextmanager
BOOTSTRAP_TIMES=[]
@contextmanager
def bootstrap_stage(name):
    started=time.perf_counter()
    try: yield
    finally: BOOTSTRAP_TIMES.append(dict(name=name,kind='setup',cpu_wall_ms=1000*(time.perf_counter()-started)))
os.environ.setdefault('HF_HUB_DISABLE_XET','1')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT','300')
REPO_URL='https://github.com/0x0shephard/latent-reasoning.git'
BASE_COMMIT='6a8d2e61950c67f012d0a9ba13ec8a70f3a25019'
REPO_DIR='/kaggle/working/latent-reasoning'
with bootstrap_stage('git_clone_fetch_checkout'):
    if not pathlib.Path(REPO_DIR).exists():
        subprocess.run(['git','clone',REPO_URL,REPO_DIR],check=True)
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin'],check=True)
    subprocess.run(['git','-C',REPO_DIR,'checkout','--detach',BASE_COMMIT],check=True)
os.chdir(REPO_DIR)
sys.path.insert(0,REPO_DIR)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==BASE_COMMIT
with bootstrap_stage('dependency_installation'):
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.52.4',
        'peft==0.15.2','datasets==3.6.0','huggingface_hub==0.32.4',
        'accelerate==1.7.0','pandas==2.2.3','pyyaml','pytest','tqdm'],check=True)
    probe=subprocess.run([sys.executable,'-c',
        'from peft.import_utils import is_torchao_available; print(is_torchao_available())'],capture_output=True,text=True)
    if probe.returncode and 'torchao' in (probe.stdout+probe.stderr):
        subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
    elif probe.returncode: raise RuntimeError(probe.stderr)

## Embedded runtime and experiment identity

In [ ]:
RUNTIME_SOURCE = '"""Detailed, opt-in diagnostics for CODI and its shared-weight explicit CoT path.\n\nClean latency is measured with instrumentation disabled. Module timings are inclusive\nstream intervals, not additive kernel totals; Chrome traces expose individual kernels.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager, nullcontext\nfrom dataclasses import dataclass\nimport csv\nimport gzip\nimport json\nimport math\nfrom pathlib import Path\nimport statistics\nimport time\nimport uuid\n\nimport torch\nfrom torch import nn\nfrom src.models.official_codi import official_codi_base_model, _normalized_official_questions\nfrom src.inference.official_codi_fast import FastCODIGeneration\n\n\nclass Timeline:\n    def __init__(self, device=\'cpu\', enabled=True):\n        self.device = torch.device(device)\n        self.enabled = enabled\n        self.context = {}\n        self.records = []\n        self.pending = []\n        self.stack = []\n        self.counter = 0\n        self.trace_id = uuid.uuid4().hex\n\n    def start(self, name, kind=\'stage\', gpu=True, **extra):\n        if not self.enabled:\n            return None\n        self.counter += 1\n        row = dict(self.context, trace_id=self.trace_id, event_id=self.counter,\n                   parent_id=self.stack[-1] if self.stack else None,\n                   name=name, kind=kind, **extra)\n        marker = torch.profiler.record_function(name)\n        marker.__enter__()\n        row[\'_wall_start\'] = time.perf_counter()\n        events = None\n        if gpu and self.device.type == \'cuda\':\n            events = (torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True))\n            events[0].record()\n        self.stack.append(self.counter)\n        return row, events, marker\n\n    def stop(self, token):\n        if token is None:\n            return\n        row, events, marker = token\n        if events:\n            events[1].record()\n        row[\'cpu_wall_ms\'] = (time.perf_counter() - row.pop(\'_wall_start\')) * 1000\n        marker.__exit__(None, None, None)\n        if self.stack and self.stack[-1] == row[\'event_id\']:\n            self.stack.pop()\n        self.pending.append((row, events))\n\n    @contextmanager\n    def span(self, name, kind=\'stage\', gpu=True, **extra):\n        token = self.start(name, kind, gpu, **extra)\n        try:\n            yield\n        finally:\n            self.stop(token)\n\n    @contextmanager\n    def metadata(self, **values):\n        previous = self.context.copy()\n        self.context.update(values)\n        try:\n            yield\n        finally:\n            self.context = previous\n\n    def resolve(self):\n        if self.device.type == \'cuda\' and self.pending:\n            torch.cuda.synchronize(self.device)\n        for row, events in self.pending:\n            row[\'cuda_stream_ms\'] = events[0].elapsed_time(events[1]) if events else None\n            self.records.append(row)\n        self.pending.clear()\n\n    @contextmanager\n    def modules(self, roots):\n        if not self.enabled:\n            yield\n            return\n        handles, stacks, seen = [], {}, set()\n        for prefix, root in roots:\n            for suffix, module in root.named_modules():\n                if id(module) in seen:\n                    continue\n                seen.add(id(module))\n                name = prefix + (\'.\' + suffix if suffix else \'\')\n                stacks[id(module)] = []\n                def pre(mod, args, label=name):\n                    token = self.start(label, kind=\'module\', inclusive=True,\n                                       module_type=type(mod).__name__)\n                    stacks[id(mod)].append(token)\n                def post(mod, args, output):\n                    self.stop(stacks[id(mod)].pop())\n                handles.append(module.register_forward_pre_hook(pre))\n                handles.append(module.register_forward_hook(post, always_call=True))\n        try:\n            yield\n        finally:\n            for handle in handles:\n                handle.remove()\n\n    def flush(self, path):\n        self.resolve()\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with gzip.open(path, \'at\', encoding=\'utf-8\') as handle:\n            for row in self.records:\n                handle.write(json.dumps(row, default=str) + \'\\n\')\n        self.records.clear()\n\n\ndef summarize_events(raw_path, output_path):\n    """Streaming moments; no need to load a large per-layer log into memory."""\n    groups = {}\n    keys = (\'mode\', \'arm\', \'protocol\', \'batch_size\', \'phase\', \'kind\', \'name\', \'module_type\')\n    with gzip.open(raw_path, \'rt\', encoding=\'utf-8\') as stream:\n        for line in stream:\n            row = json.loads(line)\n            key = tuple(str(row.get(k, \'\')) for k in keys)\n            group = groups.setdefault(key, {\'calls\': 0, \'cpu\': [], \'cuda\': []})\n            group[\'calls\'] += 1\n            for source, target in [(\'cpu_wall_ms\', \'cpu\'), (\'cuda_stream_ms\', \'cuda\')]:\n                if row.get(source) is not None:\n                    value = float(row[source])\n                    if not group[target]:\n                        group[target] = [0, 0.0, 0.0, value, value]\n                    stats = group[target]\n                    stats[0] += 1\n                    delta = value - stats[1]\n                    stats[1] += delta / stats[0]\n                    stats[2] += delta * (value - stats[1])\n                    stats[3], stats[4] = min(stats[3], value), max(stats[4], value)\n    rows = []\n    for key, group in groups.items():\n        row = dict(zip(keys, key), calls=group[\'calls\'])\n        for clock in (\'cpu\', \'cuda\'):\n            stats = group[clock]\n            if stats:\n                n, mean, m2, minimum, maximum = stats\n                row.update({f\'{clock}_mean_ms\': mean, f\'{clock}_std_ms\': math.sqrt(max(m2, 0)/max(n-1, 1)),\n                            f\'{clock}_total_ms\': mean*n, f\'{clock}_min_ms\': minimum,\n                            f\'{clock}_max_ms\': maximum})\n        rows.append(row)\n    write_csv(output_path, rows)\n    return rows\n\n\ndef write_csv(path, rows):\n    if not rows:\n        return\n    fields = list(dict.fromkeys(key for row in rows for key in row))\n    with Path(path).open(\'w\', newline=\'\', encoding=\'utf-8\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\n@dataclass\nclass PreparedBatch:\n    indices: tuple\n    ids: torch.Tensor\n    mask: torch.Tensor\n\n\ndef prepare_questions(tokenizer, questions, batch_size, timeline=None):\n    timeline = timeline or Timeline(enabled=False)\n    with timeline.span(\'question_normalization\', gpu=False):\n        questions = _normalized_official_questions(questions)\n    with timeline.span(\'question_tokenization\', gpu=False):\n        encoded = tokenizer(questions, add_special_tokens=False, padding=False)[\'input_ids\']\n    result = []\n    for start in range(0, len(encoded), batch_size):\n        part = encoded[start:start+batch_size]\n        with timeline.metadata(batch_index=start//batch_size):\n            with timeline.span(\'cpu_padding_and_tensor_allocation\', gpu=False):\n                width = max(map(len, part))\n                ids = torch.full((len(part), width), tokenizer.pad_token_id, dtype=torch.long)\n                mask = torch.zeros_like(ids)\n                for i, values in enumerate(part):\n                    if not values:\n                        raise ValueError(\'Empty question\')\n                    ids[i, -len(values):] = torch.tensor(values)\n                    mask[i, -len(values):] = 1\n            result.append(PreparedBatch(tuple(range(start, start+len(part))), ids, mask))\n    return result\n\n\ndef select_token(head, hidden, vocabulary_stop):\n    specialized = getattr(head, \'select_token\', None)\n    return specialized(hidden, vocabulary_stop=vocabulary_stop) if callable(specialized) else head(hidden)[..., :vocabulary_stop].argmax(-1)\n\n\n@torch.no_grad()\ndef decode(model, tokenizer, batches, head, *, mode, device, max_new_tokens=256,\n           latent_iterations=6, timeline=None, observer=None, forced_tokens=None):\n    """Same CODI body-only contract; explicit mode starts directly from the question.\n\n    Fixed replay evaluates every head but feeds back dense-reference token IDs.\n    Timed CUDA paths must be called inside torch.inference_mode() by the runner.\n    """\n    if mode not in (\'codi\', \'explicit_cot\'):\n        raise ValueError(mode)\n    if max_new_tokens <= 0:\n        raise ValueError(\'max_new_tokens must be positive\')\n    timeline = timeline or Timeline(device, enabled=False)\n    device = torch.device(device)\n    base = official_codi_base_model(model)\n    body, embedding = base.transformer, model.input_embeddings()\n    model.eval()\n    head.eval()\n    vocabulary_stop = int(model.eot_id)\n    eos = int(tokenizer.eos_token_id)\n    total = sum(len(b.indices) for b in batches)\n    outputs, texts, counts_out = [None]*total, [None]*total, [0]*total\n    with timeline.span(\'answer_cue_tokenization\', gpu=False):\n        cue_ids = tokenizer(\' The answer is:\', add_special_tokens=False)[\'input_ids\'] if mode == \'codi\' else []\n    for batch_number, batch in enumerate(batches):\n        with timeline.metadata(batch_index=batch_number, question_indices=list(batch.indices), token_position=-1):\n            with timeline.span(\'host_to_device_input_ids\'):\n                ids = batch.ids.to(device, non_blocking=True)\n            with timeline.span(\'host_to_device_attention_mask\'):\n                mask = batch.mask.to(device, non_blocking=True)\n            with timeline.span(\'prompt_tensor_construction\'):\n                if mode == \'codi\':\n                    bot = torch.full((len(ids),1), model.bot_id, device=device, dtype=torch.long)\n                    ids = torch.cat((ids,bot),1)\n                    mask = torch.cat((mask,torch.ones_like(bot)),1)\n            # Do not silently truncate GPT-2 context.\n            maximum = getattr(model.config, \'n_positions\', 1024) if hasattr(model, \'config\') else 1024\n            reserve = latent_iterations + 1 + len(cue_ids) if mode == \'codi\' else 0\n            if ids.shape[1] + reserve + max_new_tokens > maximum:\n                raise ValueError(\'Prompt plus generation exceeds context; reduce the configured token cap\')\n            with timeline.span(\'prefill_position_ids\'):\n                positions = mask.long().cumsum(-1)-1 if mode == \'explicit_cot\' else None\n                if positions is not None:\n                    positions.masked_fill_(mask == 0, 1)\n            with timeline.metadata(phase=\'prefill\'):\n                with timeline.span(\'transformer_prefill\'):\n                    prefill_kwargs = dict(input_ids=ids, attention_mask=mask, use_cache=True, return_dict=True)\n                    if positions is not None:\n                        prefill_kwargs[\'position_ids\'] = positions\n                    out = body(**prefill_kwargs)\n            with timeline.span(\'kv_cache_reference_update\'):\n                cache = out.past_key_values\n                hidden = out.last_hidden_state[:, -1:, :]\n            if mode == \'codi\':\n                with timeline.metadata(phase=\'latent_projection_initial\'):\n                    with timeline.span(\'latent_projector\'):\n                        latent = model.prj(hidden)\n                for step in range(latent_iterations):\n                    with timeline.metadata(phase=\'latent\', latent_step=step):\n                        with timeline.span(\'transformer_latent_pass\'):\n                            out = body(inputs_embeds=latent, past_key_values=cache, use_cache=True, return_dict=True)\n                        with timeline.span(\'kv_cache_reference_update\'):\n                            cache = out.past_key_values\n                        with timeline.span(\'latent_projector\'):\n                            latent = model.prj(out.last_hidden_state[:, -1:, :])\n                with timeline.metadata(phase=\'answer_cue\'):\n                    with timeline.span(\'answer_cue_tensor_construction\'):\n                        cue = torch.tensor([model.eot_id,*cue_ids], device=device).unsqueeze(0).expand(len(ids),-1)\n                    with timeline.span(\'answer_cue_embedding\'):\n                        cue_embedding = embedding(cue)\n                    with timeline.span(\'transformer_answer_cue\'):\n                        out = body(inputs_embeds=cue_embedding, past_key_values=cache, use_cache=True, return_dict=True)\n                    cache = out.past_key_values\n                    hidden = out.last_hidden_state[:, -1:, :]\n            with timeline.span(\'generation_buffer_allocation\'):\n                tokens = torch.full((len(ids), max_new_tokens), eos, device=device, dtype=torch.long)\n                counts = torch.zeros(len(ids), device=device, dtype=torch.long)\n                finished = torch.zeros(len(ids), device=device, dtype=torch.bool)\n            replay = None\n            if forced_tokens is not None:\n                with timeline.span(\'replay_cpu_padding\', gpu=False):\n                    seqs = [forced_tokens[i] for i in batch.indices]\n                    limit = max(map(len,seqs))\n                    if limit > max_new_tokens or any(not seq for seq in seqs):\n                        raise ValueError(\'Replay tokens must fit the generation cap\')\n                    replay_cpu = torch.full((len(ids),limit), eos, dtype=torch.long)\n                    for row, seq in enumerate(seqs):\n                        replay_cpu[row,:len(seq)] = torch.tensor(seq)\n                with timeline.span(\'host_to_device_replay_tokens\'):\n                    replay = replay_cpu.to(device)\n            else:\n                limit = max_new_tokens\n            for position in range(limit):\n                with timeline.metadata(phase=\'visible_decode\', token_position=position):\n                    if observer:\n                        with timeline.span(\'collect_hidden_state_to_cpu\'):\n                            observer(hidden[:, -1, :], ~finished, position, batch.indices)\n                    with timeline.span(\'lm_head_and_argmax\'):\n                        predicted = select_token(head, hidden[:, -1, :], vocabulary_stop)\n                    with timeline.span(\'token_selection_and_buffers\'):\n                        token = predicted if replay is None else replay[:, position]\n                        active = ~finished\n                        tokens[:,position] = torch.where(active,token,tokens[:,position])\n                        counts += active.long()\n                        finished |= active & (token == eos)\n                    with timeline.span(\'termination_check_host_sync\'):\n                        stop = position+1 == limit or bool(finished.all())\n                    if stop:\n                        break\n                    with timeline.span(\'next_token_embedding\'):\n                        embedded = embedding(token).unsqueeze(1)\n                    with timeline.span(\'decode_attention_mask_update\'):\n                        if mode == \'explicit_cot\':\n                            mask = torch.cat((mask,torch.ones((len(ids),1),device=device,dtype=mask.dtype)),1)\n                    with timeline.span(\'transformer_visible_token\'):\n                        kwargs = dict(inputs_embeds=embedded, past_key_values=cache, use_cache=True, return_dict=True)\n                        if mode == \'explicit_cot\':\n                            kwargs[\'attention_mask\'] = mask\n                            kwargs[\'position_ids\'] = (mask.long().sum(-1)-1).unsqueeze(1)\n                        out = body(**kwargs)\n                    with timeline.span(\'kv_cache_reference_update\'):\n                        cache, hidden = out.past_key_values, out.last_hidden_state\n            with timeline.span(\'device_to_host_token_buffer\'):\n                cpu_tokens = tokens.cpu()\n            with timeline.span(\'device_to_host_counts\'):\n                cpu_counts = counts.cpu().tolist()\n            with timeline.span(\'cpu_token_conversion_and_text_decode\', gpu=False):\n                for row,index in enumerate(batch.indices):\n                    count = int(cpu_counts[row])\n                    seq = tuple(int(t) for t in cpu_tokens[row,:count].tolist())\n                    outputs[index], counts_out[index] = seq, count\n                    texts[index] = tokenizer.decode(seq, skip_special_tokens=True)\n            timeline.resolve()\n    return FastCODIGeneration(tuple(texts),tuple(outputs),tuple(counts_out))\n\n\nclass DenseSelector(nn.Module):\n    def __init__(self, head, vocabulary_size):\n        super().__init__()\n        self.head = head\n        self.vocabulary_size = vocabulary_size\n    def forward(self, hidden):\n        return self.head(hidden)[..., :self.vocabulary_size]\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\nclass FixedRankHead(nn.Module):\n    def __init__(self, source, rank=96):\n        super().__init__()\n        self.vocabulary_size = source.vocabulary_size\n        self.down = nn.Linear(source.hidden_size,rank)\n        self.up = nn.Linear(rank,source.vocabulary_size)\n        with torch.no_grad():\n            self.down.weight.copy_(source.down.weight[:rank])\n            self.down.bias.copy_(source.down.bias[:rank])\n            self.up.weight.copy_(source.up.weight[:,:rank])\n            self.up.bias.copy_(source.up.bias)\n        self.requires_grad_(False)\n    def forward(self, hidden):\n        return self.up(self.down(hidden))\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\nclass CompiledSelector(nn.Module):\n    def __init__(self, source):\n        super().__init__()\n        self.core = source\n        self.requires_grad_(False)\n        object.__setattr__(self, \'compiled\', torch.compile(self._choose, mode=\'reduce-overhead\', dynamic=True, fullgraph=True))\n    def _choose(self, hidden):\n        return self.core(hidden).argmax(-1)\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self.compiled(hidden.contiguous())\n\n\ntry:\n    import triton\n    import triton.language as tl\nexcept ImportError:\n    triton = None\n\nif triton is not None:\n    @triton.jit\n    def _partial(coordinates, weights, bias, values, indices,\n                 R:tl.constexpr, V:tl.constexpr, BLOCKS:tl.constexpr,\n                 WIDE:tl.constexpr, BR:tl.constexpr, BV:tl.constexpr):\n        row, block = tl.program_id(0), tl.program_id(1)\n        vr = block*BV + tl.arange(0,BV)\n        rr = tl.arange(0,BR)\n        x = tl.load(coordinates+row*R+rr, rr<R, 0)\n        w = tl.load(weights+vr[:,None]*R+rr[None,:], (vr[:,None]<V)&(rr[None,:]<R),0)\n        b = tl.load(bias+vr, vr<V,0)\n        if WIDE:\n            # Match the eager output dtype as well as widening the arithmetic.\n            score = tl.sum(w.to(tl.float32)*x[None,:].to(tl.float32),1) + b.to(tl.float32)\n            score = score.to(b.dtype).to(tl.float32)\n        else:\n            # Historical notebook\'s numerical path, retained as an explicit control.\n            score = tl.sum(w*x[None,:],1) + b\n        score = tl.where(vr<V, score, float(\'-inf\'))\n        best = tl.argmax(score,0)\n        tl.store(values+row*BLOCKS+block,tl.max(score,0))\n        tl.store(indices+row*BLOCKS+block,block*BV+best)\n\n    @triton.jit\n    def _finish(values,indices,result,BLOCKS:tl.constexpr,B:tl.constexpr):\n        row = tl.program_id(0)\n        offsets = tl.arange(0,B)\n        vals = tl.load(values+row*BLOCKS+offsets, offsets<BLOCKS,float(\'-inf\'))\n        winner = tl.argmax(vals,0)\n        tl.store(result+row,tl.load(indices+row*BLOCKS+winner))\n\n\nclass TritonSelector(nn.Module):\n    def __init__(self, source, wide=True):\n        super().__init__()\n        if triton is None:\n            raise RuntimeError(\'Triton is unavailable\')\n        self.core, self.wide = source, wide\n        self.requires_grad_(False)\n    def select_token(self, hidden, *, vocabulary_stop):\n        x = self.core.down(hidden).contiguous()\n        w,b = self.core.up.weight,self.core.up.bias\n        batch,rank = x.shape\n        blocks = triton.cdiv(vocabulary_stop,128)\n        values = torch.empty((batch,blocks),device=x.device,dtype=torch.float32)\n        indices = torch.empty((batch,blocks),device=x.device,dtype=torch.int32)\n        result = torch.empty(batch,device=x.device,dtype=torch.long)\n        _partial[(batch,blocks)](x,w,b,values,indices,rank,vocabulary_stop,blocks,self.wide,\n                                 triton.next_power_of_2(rank),128,num_warps=4)\n        _finish[(batch,)](values,indices,result,blocks,triton.next_power_of_2(blocks),num_warps=4)\n        return result\n\n\ndef summarize_timings(rows):\n    groups = {}\n    keys = (\'mode\',\'arm\',\'protocol\',\'batch_size\')\n    for row in rows:\n        groups.setdefault(tuple(row[k] for k in keys),[]).append(row)\n    summary = []\n    for key, group in groups.items():\n        values = [x[\'wall_ms_per_question\'] for x in group]\n        summary.append(dict(zip(keys,key), measurements=len(group),\n            mean_ms_per_question=statistics.mean(values), median_ms_per_question=statistics.median(values),\n            std_ms_per_question=statistics.stdev(values) if len(values)>1 else 0.0,\n            min_ms_per_question=min(values),max_ms_per_question=max(values),\n            total_questions=sum(x[\'questions\'] for x in group),\n            mean_generated_tokens=statistics.mean(x[\'visible_tokens\']/x[\'questions\'] for x in group)))\n    return summary\n'
RUNTIME_SHA256 = '1e95250f0928a893fc390edf4091e2cae53ba52da8d7a54d166448accf883f3a'
EXPERIMENT_SOURCE_SHA256 = 'e98e981e1d06eb5286a320a98686fae73d9b07d50aa0fd2902f88787c8790927'
with bootstrap_stage('runtime_import'):
    assert hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest()==RUNTIME_SHA256
    runtime_path=pathlib.Path('/kaggle/working/dual_global_head_runtime.py')
    runtime_path.write_text(RUNTIME_SOURCE)
    spec=importlib.util.spec_from_file_location('dual_global_head_runtime',runtime_path)
    runtime=importlib.util.module_from_spec(spec)
    sys.modules[spec.name]=runtime
    spec.loader.exec_module(runtime)
    import torch
    import pandas as pd
    from dataclasses import asdict
    from src.mech.global_low_rank_head import (
        NestedLowRankVocabularyHead,activation_whitened_factors,distil_nested_head,evaluate_nested_head)
    from src.models.official_codi import (
        build_official_codi_gpt2,download_official_checkpoint,load_official_checkpoint,official_codi_base_model)
    from src.inference.official_codi_fast import (
        generate_official_codi_fast,prepare_official_codi_batches,merge_official_codi_lora_)
    from src.data.answer_extract import answers_match,normalize_gold
    from src.utils.config import load_config
assert torch.cuda.is_available(),'Enable a Kaggle GPU'
device=torch.device('cuda')
torch.manual_seed(SEED); random.seed(SEED)
config=dict(smoke=SMOKE,seed=SEED,modes=MODES,fit=FIT_QUESTIONS,selection=SELECT_QUESTIONS,
    recovery=RECOVERY_QUESTIONS,state_caps=[MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES],
    epochs=[CLEAN_EPOCHS,RECOVERY_EPOCHS],distill_batch=DISTILL_BATCH_SIZE,
    collect_batch=COLLECT_BATCH_SIZE,quality_batch=QUALITY_BATCH_SIZE,quality=QUALITY_QUESTIONS,
    token_caps=MAX_NEW_TOKENS,timing_questions=TIMING_QUESTIONS,batches=TIMING_BATCH_SIZES,
    repeats=TIMING_REPEATS,profile_questions=PROFILE_QUESTIONS,profile_repeats=PROFILE_REPEATS,
    operator_trace_questions=OPERATOR_TRACE_QUESTIONS,compile=TRY_COMPILE,triton=TRY_TRITON,
    base=BASE_COMMIT,source=EXPERIMENT_SOURCE_SHA256,torch=torch.__version__,cuda=torch.version.cuda,
    triton_version=getattr(runtime.triton,'__version__',None),gpu=torch.cuda.get_device_name(0))
manifest=json.dumps(config,sort_keys=True,indent=2)
RUN_ID=hashlib.sha256(manifest.encode()).hexdigest()[:16]
RUN_DIR=pathlib.Path(OUTPUT_ROOT)/('smoke_' if SMOKE else 'full_')/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=True)
if RESUME_DIR:
    previous=pathlib.Path(RESUME_DIR)
    assert (previous/'manifest.json').read_text()==manifest,'Resume settings/source/runtime mismatch'
    shutil.copytree(previous,RUN_DIR,dirs_exist_ok=True)
if (RUN_DIR/'manifest.json').exists(): assert (RUN_DIR/'manifest.json').read_text()==manifest
(RUN_DIR/'manifest.json').write_text(manifest)
setup=runtime.Timeline(device)
setup.records.extend(BOOTSTRAP_TIMES)
setup.context.update(mode='setup',protocol='setup')
setup.flush(RUN_DIR/'setup_events.jsonl.gz')
def save_json(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); temp.write_text(json.dumps(value,indent=2,default=str)); temp.replace(path)
def save_pt(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); torch.save(value,temp); temp.replace(path)
print('Output:',RUN_DIR)
print(json.dumps(config,indent=2))

## Download data and the shared checkpoint
Canonical GSM8K train supplies disjoint fit, selection, recovery, timing and warmup
questions, shared by both modes. Test labels never select weights or settings. Setup
costs are separated from warm inference; cached downloads naturally finish faster.

In [ ]:
from datasets import load_dataset
DATA_REVISION='3101c7d5072418e28b9008a6636bde82a006892c'
url=f'https://raw.githubusercontent.com/openai/grade-school-math/{DATA_REVISION}/grade_school_math/data/'
with setup.span('gsm8k_train_download_and_parse',gpu=False):
    train=load_dataset('json',data_files={'train':url+'train.jsonl'},split='train')
with setup.span('gsm8k_test_download_and_parse',gpu=False):
    test=load_dataset('json',data_files={'test':url+'test.jsonl'},split='test')
with setup.span('dataset_normalization_and_partitioning',gpu=False):
    unique={}
    for row in train:
        key=' '.join(row['question'].casefold().split())
        unique.setdefault(key,dict(question=str(row['question']),gold=str(normalize_gold(row['answer'],'gsm8k_main'))))
    rows=list(unique.values()); random.Random(SEED).shuffle(rows)
    cuts=[FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS,TIMING_QUESTIONS,32]
    splits={}; start=0
    for name,size in zip(['fit','selection','recovery','timing','warmup'],cuts):
        splits[name]=rows[start:start+size]; start+=size
    assert start<=len(rows)
    test_rows=[dict(question=str(r['question']),gold=str(normalize_gold(r['answer'],'gsm8k_main'))) for r in test]
    assert len(test_rows)==1319
    selected={' '.join(r['question'].casefold().split()) for part in splits.values() for r in part}
    assert not selected & {' '.join(r['question'].casefold().split()) for r in test_rows}
    test_rows=test_rows[:QUALITY_QUESTIONS]
    save_json(RUN_DIR/'partitions.json',dict(splits=splits,test_questions=[r['question'] for r in test_rows]))
with setup.span('config_load',gpu=False): cfg=load_config('configs/official_codi_gpt2.yaml')
with setup.span('codi_checkpoint_download_and_hash',gpu=False):
    checkpoint=download_official_checkpoint(repo_id=cfg.checkpoint.repo_id,revision=cfg.checkpoint.revision,
        filename=cfg.checkpoint.filename,expected_sha256=cfg.checkpoint.sha256)
with setup.span('gpt2_tokenizer_download_and_model_construction',gpu=False):
    model,tokenizer=build_official_codi_gpt2(base_model=cfg.model.base_model,base_revision=cfg.model.base_revision,
        dtype=torch.float32,settings=cfg.model)
with setup.span('checkpoint_load_and_verification',gpu=False):
    load_report=load_official_checkpoint(model,checkpoint,expected_sha256=cfg.checkpoint.sha256)
with setup.span('model_host_to_device_fp32'): model.requires_grad_(False).to(device).eval()
base=official_codi_base_model(model)
full_head=base.get_output_embeddings()
weight=full_head.weight[:model.eot_id].detach()
bias=None if getattr(full_head,'bias',None) is None else full_head.bias[:model.eot_id].detach()
setup.flush(RUN_DIR/'setup_events.jsonl.gz')
print('Shared GPT-2 checkpoint:',load_report.checkpoint_sha256)

## Verify both generation paths before fitting

In [ ]:
parity_questions=[r['question'] for r in splits['selection'][:4]]
parity={}
with torch.no_grad():
    for mode in MODES:
        prepared=runtime.prepare_questions(tokenizer,parity_questions,4)
        observed=runtime.decode(model,tokenizer,prepared,full_head,mode=mode,device=device,
            max_new_tokens=MAX_NEW_TOKENS[mode],latent_iterations=6)
        if mode=='codi':
            reference=generate_official_codi_fast(model,tokenizer,
                prepare_official_codi_batches(tokenizer,parity_questions,batch_size=4,length_bucketed=False),
                latent_iterations=6,max_new_tokens=MAX_NEW_TOKENS[mode],device=device,answer_cue='The answer is:')
            expected=reference.token_ids
        else:
            from transformers import LogitsProcessor,LogitsProcessorList
            class VocabularyBoundary(LogitsProcessor):
                def __call__(self,input_ids,scores):
                    scores[:,int(model.eot_id):]=float('-inf'); return scores
            batch=prepared[0]
            generated=base.generate(input_ids=batch.ids.to(device),attention_mask=batch.mask.to(device),
                do_sample=False,max_new_tokens=MAX_NEW_TOKENS[mode],pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,logits_processor=LogitsProcessorList([VocabularyBoundary()]))
            expected=[]
            for seq in generated[:,batch.ids.shape[1]:].cpu().tolist():
                if tokenizer.eos_token_id in seq: seq=seq[:seq.index(tokenizer.eos_token_id)+1]
                expected.append(tuple(seq))
            expected=tuple(expected)
        parity[mode]=dict(examples=len(expected),exact=observed.token_ids==expected)
        assert parity[mode]['exact'],f'{mode}: custom decoder differs from reference; stop before fitting'
save_json(RUN_DIR/'decoder_parity.json',parity)
print(parity)

## Fit one global head for each reasoning mode
Four clean epochs plus two recovery epochs; the rank-64 prefix generates recovery
states. Validation agreement, then KL, selects checkpoints. Training is FP32;
deployment uses FP16 with merged LoRA, matching the previous efficiency notebook.

In [ ]:
def collect(mode,head,population,cap,tag):
    path=RUN_DIR/f'states_{mode}_{tag}.pt'
    if path.exists():
        with setup.span('state_cache_disk_load',gpu=False,mode=mode,tag=tag):
            return torch.load(path,map_location='cpu',weights_only=False)
    chunks=[]; positions=[]
    def observe(hidden,active,position,indices):
        chunks.append(hidden[active].detach().cpu().float())
        positions.extend([position]*int(active.sum()))
    questions=[r['question'] for r in splits[population]]
    batches=runtime.prepare_questions(tokenizer,questions,COLLECT_BATCH_SIZE)
    with setup.span('trajectory_collection',mode=mode,population=population):
        runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,
                       max_new_tokens=MAX_NEW_TOKENS[mode],observer=observe)
    values=torch.cat(chunks)
    indices=torch.randperm(len(values),generator=torch.Generator().manual_seed(SEED))[:cap]
    result=dict(states=values[indices].clone(),positions=torch.tensor(positions)[indices],observed_states=len(values))
    with setup.span('state_cache_disk_save',gpu=False,mode=mode,tag=tag): save_pt(path,result)
    return result

def evaluate_positions(head,bundle):
    result={}
    for rank in RANKS:
        result[str(rank)]={}
        for label,mask in [('all',torch.ones(len(bundle['states']),dtype=torch.bool)),
                           ('p0',bundle['positions']==0),('p1',bundle['positions']==1),('p2plus',bundle['positions']>=2)]:
            result[str(rank)][label]=dict(states=int(mask.sum()),**(evaluate_nested_head(
                head,bundle['states'][mask],weight,readout_bias=bias,rank=rank,batch_size=DISTILL_BATCH_SIZE)
                if mask.any() else {}))
    return result

for mode in MODES:
    artifact=RUN_DIR/f'global_head_{mode}.pt'
    if artifact.exists(): print('Reuse fitted head:',mode); continue
    print('Collect/fitting:',mode,flush=True)
    fit=collect(mode,full_head,'fit',MAX_FIT_STATES,'fit')
    selection=collect(mode,full_head,'selection',MAX_SELECT_STATES,'selection')
    with setup.span('activation_whitened_initialization',mode=mode):
        centre,down,up,out_bias,init=activation_whitened_factors(fit['states'],weight,96,
            readout_bias=bias,seed=SEED,compute_device=device)
        head=NestedLowRankVocabularyHead.from_whitened_factors(centre,down,up,out_bias,RANKS).to(device)
    initial_metrics=evaluate_positions(head,selection)
    with setup.span('clean_distillation',mode=mode):
        clean=distil_nested_head(head,fit['states'],selection['states'],weight,readout_bias=bias,
            epochs=CLEAN_EPOCHS,batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED)
    head.disable_adaptive(); head.set_rank(64)
    recovery=collect(mode,head,'recovery',MAX_RECOVERY_STATES,'onpolicy')
    with setup.span('recovery_distillation',mode=mode):
        recovered=distil_nested_head(head,torch.cat((fit['states'],recovery['states'])),
            selection['states'],weight,readout_bias=bias,epochs=RECOVERY_EPOCHS,
            batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED+1)
    report=dict(mode=mode,initialization=asdict(init),clean=asdict(clean),recovery=asdict(recovered),
                initial=initial_metrics,final=evaluate_positions(head,selection),
                fit_states=len(fit['states']),recovery_states=len(recovery['states']))
    with setup.span('trained_head_disk_save',gpu=False,mode=mode):
        save_pt(artifact,dict(state_dict={k:v.detach().cpu().clone() for k,v in head.state_dict().items()},report=report))
        save_json(artifact.with_suffix('.json'),report)
    setup.flush(RUN_DIR/'setup_events.jsonl.gz')
    del head,fit,selection,recovery,centre,down,up,out_bias
    gc.collect(); torch.cuda.empty_cache()
with setup.span('lora_merge'): merge_official_codi_lora_(model)
with setup.span('model_fp16_conversion'): model.to(dtype=torch.float16).eval()
base=official_codi_base_model(model); full_head=base.get_output_embeddings()
del weight,bias
setup.flush(RUN_DIR/'setup_events.jsonl.gz')

## Deployment measurement runner
Raw rows retain mode, head, protocol, repeat, batch/question IDs, token counts and wall
time. Clean runs install no module hooks or per-layer CUDA events. Arm order is shuffled
inside each repeat. Compilation and real-shape warmups occur before timing. Optional
backend failures are exported explicitly. Fixed replay is never used for accuracy.

In [ ]:
def local_batch(batch):
    return runtime.PreparedBatch(tuple(range(len(batch.indices))),batch.ids,batch.mask)

@torch.inference_mode()
def timed_batch(mode,head,batch,reference=None,timeline=None):
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    allocated=torch.cuda.memory_allocated(); started=time.perf_counter()
    result=runtime.decode(model,tokenizer,[local_batch(batch)],head,mode=mode,device=device,
        max_new_tokens=MAX_NEW_TOKENS[mode],forced_tokens=reference,timeline=timeline)
    torch.cuda.synchronize(); wall=1000*(time.perf_counter()-started)
    return result,dict(wall_ms=wall,wall_ms_per_question=wall/len(batch.indices),
        questions=len(batch.indices),visible_tokens=result.generated_token_count,
        peak_increment_bytes=max(0,torch.cuda.max_memory_allocated()-allocated))

def make_arms(mode):
    with setup.span('fitted_head_disk_load',gpu=False,mode=mode):
        payload=torch.load(RUN_DIR/f'global_head_{mode}.pt',map_location='cpu',weights_only=False)
    nested=NestedLowRankVocabularyHead(int(model.config.hidden_size),int(model.eot_id),RANKS)
    nested.load_state_dict(payload['state_dict'])
    with setup.span('fitted_head_host_to_device_fp16',mode=mode):
        eager=runtime.FixedRankHead(nested,96).to(device=device,dtype=torch.float16).eval()
    arms={'dense':runtime.DenseSelector(full_head,int(model.eot_id)),'rank96_eager':eager}
    errors={}; constructors={}
    if TRY_COMPILE: constructors['rank96_compiled']=lambda:runtime.CompiledSelector(eager)
    if TRY_TRITON:
        constructors['rank96_triton_legacy']=lambda:runtime.TritonSelector(eager,wide=False)
        constructors['rank96_triton_fp32']=lambda:runtime.TritonSelector(eager,wide=True)
    warm=[r['question'] for r in splits['warmup']]
    for name,constructor in constructors.items():
        try:
            with setup.span('optional_backend_construction_and_warmup',mode=mode,arm=name):
                head=constructor().eval()
                for batch_size in TIMING_BATCH_SIZES:
                    batch=runtime.prepare_questions(tokenizer,warm[:batch_size],batch_size)[0]
                    timed_batch(mode,head,batch)
                arms[name]=head
        except Exception as error: errors[name]=repr(error)
    for name in ('dense','rank96_eager'):
        for batch_size in TIMING_BATCH_SIZES:
            timed_batch(mode,arms[name],runtime.prepare_questions(tokenizer,warm[:batch_size],batch_size)[0])
    save_json(RUN_DIR/f'backend_status_{mode}.json',dict(available=list(arms),errors=errors))
    return arms

ALL_TIMINGS=[]; ALL_QUALITY=[]; HEAD_TIMINGS=[]; PROFILE_OVERHEAD=[]
for mode in MODES:
    print('Benchmark:',mode,flush=True)
    mode_dir=RUN_DIR/mode; mode_dir.mkdir(exist_ok=True)
    if (mode_dir/'completed.json').exists():
        ALL_TIMINGS.extend(json.loads((mode_dir/'clean_raw.json').read_text()))
        ALL_QUALITY.extend(json.loads((mode_dir/'quality_summary.json').read_text()))
        HEAD_TIMINGS.extend(json.loads((mode_dir/'head_raw.json').read_text()))
        PROFILE_OVERHEAD.extend(json.loads((mode_dir/'profile_overhead.json').read_text()))
        continue
    path=mode_dir/'detailed_events.jsonl.gz'
    if path.exists(): path.unlink()  # restart only this incomplete mode's diagnostic log
    arms=make_arms(mode)
    raw=[]; quality=[]; head_raw=[]; overhead=[]
    timing_questions=[r['question'] for r in splits['timing']]
    with setup.span('timing_prompt_preparation',gpu=False,mode=mode):
        prepared={b:runtime.prepare_questions(tokenizer,timing_questions,b) for b in TIMING_BATCH_SIZES}
    with torch.inference_mode():
        references={}
        for batch_size,batches in prepared.items():
            result=runtime.decode(model,tokenizer,batches,arms['dense'],mode=mode,device=device,
                                  max_new_tokens=MAX_NEW_TOKENS[mode])
            references[batch_size]=result.token_ids
        save_json(mode_dir/'dense_timing_reference_tokens.json',references)
        # Real FP16 deployment states, not the earlier FP32 fitting-state cache.
        state_chunks=[]
        def diagnostic_observer(hidden,active,position,indices):
            state_chunks.append(hidden[active].detach().cpu())
        diagnostic_questions=[r['question'] for r in splits['warmup'][:8]]
        runtime.decode(model,tokenizer,runtime.prepare_questions(tokenizer,diagnostic_questions,8),
            arms['dense'],mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode],observer=diagnostic_observer)
        pool=torch.cat(state_chunks)
        hidden=pool[:min(512,len(pool))].to(device=device,dtype=torch.float16)
        eager_scores=arms['rank96_eager'](hidden).float(); expected=eager_scores.argmax(-1)
        fidelity={}
        for name,head in arms.items():
            tokens=runtime.select_token(head,hidden,int(model.eot_id))
            regret=eager_scores.max(-1).values-eager_scores.gather(1,tokens[:,None]).squeeze(1)
            fidelity[name]=dict(states=len(hidden),agreement_with_eager=float((tokens==expected).float().mean()),
                mean_eager_score_regret=float(regret.mean()),max_eager_score_regret=float(regret.max()))
        save_json(mode_dir/'selector_fidelity.json',fidelity)
        for batch_size in TIMING_BATCH_SIZES:
            sample=hidden[torch.arange(batch_size,device=device)%len(hidden)].contiguous()
            for name,head in arms.items():
                for _ in range(20): runtime.select_token(head,sample,int(model.eot_id))
                torch.cuda.synchronize()
                for repeat in range(TIMING_REPEATS):
                    begin,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)
                    begin.record(); started=time.perf_counter()
                    for _ in range(100): runtime.select_token(head,sample,int(model.eot_id))
                    end.record(); torch.cuda.synchronize()
                    head_raw.append(dict(mode=mode,arm=name,batch_size=batch_size,repeat=repeat,
                        cuda_stream_ms_per_call=begin.elapsed_time(end)/100,
                        wall_ms_per_call=1000*(time.perf_counter()-started)/100,calls_per_measurement=100))
        for protocol in ('free_generation','fixed_replay'):
            for batch_size,batches in prepared.items():
                for repeat in range(TIMING_REPEATS):
                    order=list(arms); random.Random(SEED+repeat+batch_size).shuffle(order)
                    for name in order:
                        for batch_index,batch in enumerate(batches):
                            ref=[references[batch_size][i] for i in batch.indices] if protocol=='fixed_replay' else None
                            result,row=timed_batch(mode,arms[name],batch,ref)
                            raw.append(dict(mode=mode,arm=name,protocol=protocol,batch_size=batch_size,
                                actual_batch_size=len(batch.indices),repeat=repeat,batch_index=batch_index,
                                question_indices=list(batch.indices),token_counts=list(result.generated_token_counts),**row))
                        save_json(mode_dir/'clean_raw.json',raw)
                        print(mode,protocol,batch_size,repeat,name,flush=True)
        for name,head in arms.items():
            batches=runtime.prepare_questions(tokenizer,[r['question'] for r in test_rows],QUALITY_BATCH_SIZE)
            result=runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,max_new_tokens=MAX_NEW_TOKENS[mode])
            records=[]
            for i,(example,text,tokens) in enumerate(zip(test_rows,result.texts,result.token_ids)):
                records.append(dict(index=i,question=example['question'],gold=example['gold'],text=text,
                    token_ids=list(tokens),correct=bool(answers_match(text,example['gold'])),
                    eos_terminated=bool(tokens and tokens[-1]==tokenizer.eos_token_id),
                    answer_cue_present='the answer is:' in text.casefold()))
            save_json(mode_dir/f'quality_{name}.json',records)
            quality.append(dict(mode=mode,arm=name,examples=len(records),correct=sum(r['correct'] for r in records),
                accuracy=sum(r['correct'] for r in records)/len(records),
                completed_correct=sum(r['correct'] and r['eos_terminated'] for r in records)/len(records),
                truncated_fraction=sum(not r['eos_terminated'] for r in records)/len(records),
                mean_generated_tokens=result.generated_token_count/len(records)))
    for name,head in arms.items():
        for repeat in range(PROFILE_REPEATS):
            trace=runtime.Timeline(device)
            trace.context.update(mode=mode,arm=name,protocol='instrumented_fixed_replay',batch_size=1,repeat=repeat)
            with trace.span('profile_prompt_preparation',gpu=False):
                batches=runtime.prepare_questions(tokenizer,timing_questions[:PROFILE_QUESTIONS],1,trace)
            for i,batch in enumerate(batches):
                ref=[references[1][i]]
                _,clean=timed_batch(mode,head,batch,ref)
                roots=[('transformer',base.transformer),('latent_projector',model.prj)]
                if name in ('dense','rank96_eager'): roots.append(('head',head))
                with trace.metadata(profile_question=i):
                    with trace.modules(roots): _,instrumented=timed_batch(mode,head,batch,ref,trace)
                overhead.append(dict(mode=mode,arm=name,repeat=repeat,question_index=i,
                    clean_wall_ms=clean['wall_ms'],instrumented_wall_ms=instrumented['wall_ms'],
                    instrumentation_ratio=instrumented['wall_ms']/clean['wall_ms']))
                trace.flush(mode_dir/'detailed_events.jsonl.gz')
        if OPERATOR_TRACE_QUESTIONS:
            from torch.profiler import profile,ProfilerActivity
            try:
                batches=runtime.prepare_questions(tokenizer,timing_questions[:OPERATOR_TRACE_QUESTIONS],1)
                with torch.inference_mode(),profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA],
                                                    record_shapes=True,profile_memory=True) as prof:
                    for i,batch in enumerate(batches): timed_batch(mode,head,batch,[references[1][i]])
                with setup.span('chrome_trace_export',gpu=False,mode=mode,arm=name):
                    prof.export_chrome_trace(str(mode_dir/f'operators_{name}.json'))
                    op_rows=[]
                    for event in prof.events():
                        op_rows.append(dict(name=event.name,device_type=str(event.device_type),
                            cpu_total_us=event.cpu_time_total,cpu_self_us=event.self_cpu_time_total,
                            device_total_us=getattr(event,'device_time_total',None),
                            device_self_us=getattr(event,'self_device_time_total',None),input_shapes=str(event.input_shapes)))
                    runtime.write_csv(mode_dir/f'operator_events_{name}.csv',op_rows)
                    (mode_dir/f'operator_averages_{name}.txt').write_text(
                        prof.key_averages(group_by_input_shape=True).table(sort_by='self_cuda_time_total',row_limit=200))
            except Exception as error: save_json(mode_dir/f'profiler_error_{name}.json',dict(error=repr(error)))
    save_json(mode_dir/'clean_raw.json',raw)
    save_json(mode_dir/'head_raw.json',head_raw)
    save_json(mode_dir/'quality_summary.json',quality)
    save_json(mode_dir/'profile_overhead.json',overhead)
    runtime.summarize_events(mode_dir/'detailed_events.jsonl.gz',mode_dir/'layer_stage_averages.csv')
    ALL_TIMINGS.extend(raw); ALL_QUALITY.extend(quality); HEAD_TIMINGS.extend(head_raw); PROFILE_OVERHEAD.extend(overhead)
    setup.flush(RUN_DIR/'setup_events.jsonl.gz')
    save_json(mode_dir/'completed.json',dict(complete=True))
    del arms,head,hidden,eager_scores,pool,state_chunks
    gc.collect(); torch.cuda.empty_cache()

## Averages, uncertainty, and individual measurements

In [ ]:
summary=runtime.summarize_timings(ALL_TIMINGS)
frame=pd.DataFrame(summary)
for index,row in frame.iterrows():
    dense=frame[(frame['mode']==row['mode'])&(frame.protocol==row.protocol)&
                (frame.batch_size==row.batch_size)&(frame.arm=='dense')].iloc[0]
    frame.loc[index,'mean_speedup_vs_dense']=dense.mean_ms_per_question/row.mean_ms_per_question
    frame.loc[index,'median_speedup_vs_dense']=dense.median_ms_per_question/row.median_ms_per_question
frame.to_csv(RUN_DIR/'timing_averages.csv',index=False)
runtime.write_csv(RUN_DIR/'timing_individual.csv',ALL_TIMINGS)
for quality_row in ALL_QUALITY:
    dense_accuracy=next(r['accuracy'] for r in ALL_QUALITY if r['mode']==quality_row['mode'] and r['arm']=='dense')
    quality_row['accuracy_retention_vs_dense']=quality_row['accuracy']/dense_accuracy if dense_accuracy else None
runtime.write_csv(RUN_DIR/'quality_summary.csv',ALL_QUALITY)
runtime.write_csv(RUN_DIR/'head_individual.csv',HEAD_TIMINGS)
runtime.write_csv(RUN_DIR/'profiling_overhead.csv',PROFILE_OVERHEAD)
head_frame=pd.DataFrame(HEAD_TIMINGS)
head_frame.groupby(['mode','arm','batch_size'])[['wall_ms_per_call','cuda_stream_ms_per_call']].agg(
    ['count','mean','median','std','min','max']).to_csv(RUN_DIR/'head_averages.csv')
# Descriptive paired bootstrap over repeat totals for the same question population.
uncertainty=[]
raw_frame=pd.DataFrame(ALL_TIMINGS)
for (mode,protocol,batch_size),group in raw_frame.groupby(['mode','protocol','batch_size']):
    totals=group.groupby(['arm','repeat']).wall_ms.sum().unstack(0)
    gen=torch.Generator().manual_seed(SEED)
    indices=torch.randint(len(totals),(2000,len(totals)),generator=gen)
    for arm in totals.columns:
        a=torch.tensor(totals['dense'].to_numpy())[indices].mean(1)
        b=torch.tensor(totals[arm].to_numpy())[indices].mean(1)
        ratio=a/b
        uncertainty.append(dict(mode=mode,protocol=protocol,batch_size=batch_size,arm=arm,
            paired_repeat_speedup_ci95_low=float(ratio.quantile(.025)),
            paired_repeat_speedup_ci95_high=float(ratio.quantile(.975)),repeats=len(totals)))
runtime.write_csv(RUN_DIR/'timing_speedup_intervals.csv',uncertainty)
runtime.summarize_events(RUN_DIR/'setup_events.jsonl.gz',RUN_DIR/'setup_averages.csv')
display(frame)
display(pd.DataFrame(ALL_QUALITY))
print('Individual measurements:'); display(raw_frame.head(20))
print('Layer/stage example:'); display(pd.read_csv(RUN_DIR/MODES[0]/'layer_stage_averages.csv').head(30))
profiler_errors=[str(p.relative_to(RUN_DIR)) for p in RUN_DIR.glob('*/profiler_error_*.json')]
if profiler_errors: print('Some operator traces failed; see:',profiler_errors)
save_json(RUN_DIR/'completed.json',dict(complete=True,smoke=SMOKE,source=EXPERIMENT_SOURCE_SHA256,
    operator_profiling_complete=not profiler_errors,profiler_errors=profiler_errors))
print('Download output folder:',RUN_DIR)

## Inspecting and interpreting output

- `timing_averages.csv`: clean mean, median, SD, range, and speedup for each mode/head/batch/protocol.
- `timing_individual.csv`: each clean **batch invocation**, with repeat and question IDs;
  batch 1 gives individual-question latency. Larger batches give batch latency, not
  independent latency for each simultaneously processed question.
- `timing_speedup_intervals.csv`: descriptive paired-repeat bootstrap intervals.
- `head_averages.csv` / `head_individual.csv`: isolated selector timing including argmax;
  each microbenchmark row averages 100 calls, explicitly recorded in the raw file.
  Individual head-call spans are in the detailed diagnostic log.
- `<mode>/layer_stage_averages.csv`: each numbered transformer layer, attention/MLP
  submodule, norm, embedding, projector, and explicitly instrumented runtime stage.
- `<mode>/detailed_events.jsonl.gz`: **individual calls** with layer name, phase,
  question, token position, repeat, parent event, CPU wall time and CUDA stream time.
- `<mode>/operators_<arm>.json`: Chrome/Perfetto CPU/CUDA operator and kernel timeline.
- `<mode>/operator_events_<arm>.csv` / `operator_averages_<arm>.txt`: raw and averaged operators.
- `setup_events.jsonl.gz` / `setup_averages.csv`: downloads, parsing, state I/O,
  model loading/movement, fitting, warmup, and trace exports.
- `profiling_overhead.csv`: runtime change introduced by instrumentation.
- `<mode>/quality_<arm>.json`: every generated answer/CoT, token IDs and termination status.
- `<mode>/selector_fidelity.json`: numerical agreement/regret on real states.

**Never sum inclusive parent and child times.** CUDA events measure stream intervals
including idle gaps; CPU module times usually measure dispatch, not GPU execution.
CPU-only stages have no CUDA timing. Profiler device events expose actual kernels.
Functional operations inside a layer appear in operator traces even without a separate
`nn.Module`. Compiled heads are timed as units; their kernels appear in operator traces,
without hooks inside the compiled head that would alter compilation.

Profiling uses a fixed sample because logging every operation over the full test would
create huge traces and heavily distort runtime. Increase `PROFILE_QUESTIONS`,
`PROFILE_REPEATS`, or `OPERATOR_TRACE_QUESTIONS` before running for broader coverage.
All clean timing repetitions are saved individually. Quality uses 1,319 questions by
default. Explicit generation stopped at its token cap is reported as truncated.

Resume: attach the prior output and point `RESUME_DIR` at the exact matching run folder.
Fitted heads and fully completed mode benchmarks are reused; an interrupted fit or mode
benchmark restarts. Keep the same settings/source/runtime. Fresh runs need no attachments.